# TARGETED WIENER RERUN

**File**: `06_visa_padim_clean_wiener.ipynb`

This notebook is a surgically stripped-down version of the original pipeline.
It is designed solely to regenerate the misspecified Wiener deconvolution rows
using the corrected, per-severity PSF parameters.

**It ONLY executes:**
- `gaussian_blur` and `motion_blur`
- `mild` and `moderate` severities
- Wiener deconvolution rescue

All other corruptions, severities, and rescue methods have been removed for speed.

In [ ]:
# ---------------------------------------------------------------------------
# 2. Embedded Corruption & Rescue Functions (Stripped down for Wiener rerun)
# ---------------------------------------------------------------------------

def apply_gaussian_blur(image, sigma, kernel_size):
    t = A.GaussianBlur(blur_limit=(kernel_size, kernel_size), sigma_limit=(sigma, sigma), p=1.0)
    return t(image=image)["image"]

def apply_motion_blur(image, kernel_size):
    t = A.MotionBlur(blur_limit=(kernel_size, kernel_size), p=1.0)
    return t(image=image)["image"]

def apply_corruption(image, ctype, severity, config, seed=42):
    params = config["corruptions"][ctype][severity]
    if ctype == "gaussian_blur": return apply_gaussian_blur(image, params["sigma"], params["kernel_size"])
    elif ctype == "motion_blur": return apply_motion_blur(image, params["kernel_size"])
    else: raise ValueError(f"Unknown corruption type: {ctype}")

def apply_wiener_deconv(image: np.ndarray, sigma: float, kernel_size: int, balance: float = 0.1) -> np.ndarray:
    import cv2, numpy as np
    from skimage.restoration import wiener
    psf = np.zeros((kernel_size, kernel_size))
    center = kernel_size // 2
    psf[center, center] = 1.0
    psf = cv2.GaussianBlur(psf, (kernel_size, kernel_size), sigmaX=sigma, sigmaY=sigma)
    psf /= psf.sum()
    out = np.zeros_like(image, dtype=np.float64)
    for i in range(3):
        out[:, :, i] = wiener(image[:, :, i] / 255.0, psf, balance, clip=False)
    out = np.clip(out * 255, 0, 255).astype(np.uint8)
    return out

def apply_motion_wiener_deconv(image: np.ndarray, kernel_size: int, balance: float = 0.1) -> np.ndarray:
    import cv2, numpy as np
    from skimage.restoration import wiener
    psf = np.zeros((kernel_size, kernel_size))
    psf[kernel_size // 2, :] = 1.0 / kernel_size
    angle = 0
    M = cv2.getRotationMatrix2D((kernel_size / 2, kernel_size / 2), angle, 1)
    psf = cv2.warpAffine(psf, M, (kernel_size, kernel_size))
    psf /= psf.sum()
    out = np.zeros_like(image, dtype=np.float64)
    for i in range(3):
        out[:, :, i] = wiener(image[:, :, i] / 255.0, psf, balance, clip=False)
    out = np.clip(out * 255, 0, 255).astype(np.uint8)
    return out

def get_rescue_map(severity, config):
    gauss_p = config["corruptions"]["gaussian_blur"][severity]
    motion_p = config["corruptions"]["motion_blur"][severity]
    return {
        "gaussian_blur": [("Wiener", lambda img, s=gauss_p["sigma"], k=gauss_p["kernel_size"]:
                           apply_wiener_deconv(img, sigma=s, kernel_size=k))],
        "motion_blur": [("Wiener (Motion PSF)", lambda img, k=motion_p["kernel_size"]:
                         apply_motion_wiener_deconv(img, kernel_size=k))]
    }
